# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
58,0.86,588.0,294.0,147.0,7.0,0.1,26.700
710,0.66,759.5,318.5,220.5,3.5,0.4,16.725
543,0.82,612.5,318.5,147.0,7.0,0.4,31.095
663,0.66,759.5,318.5,220.5,3.5,0.4,16.560
676,0.90,563.5,318.5,122.5,7.0,0.4,35.215


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

MSE ortalama kare hatayı aldığı için büyük hata daha çok göze çarpacaktır.
    

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [3]:
from sklearn.preprocessing import StandardScaler


X = data.drop(columns=["Average Temperature"])  
y = data["Average Temperature"]                  


scaler = StandardScaler()


X_scaled = scaler.fit_transform(X)

### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [5]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_validate

model_mse = SGDRegressor(
    loss="squared_error",  
    random_state=42        
)

cv_results = cross_validate(
    model_mse,
    X_scaled,
    y,
    cv=10,                                   
    scoring=["r2", "neg_max_error"]   
)

r2 = cv_results["test_r2"].mean()
max_error_celsius = -cv_results["test_neg_max_error"].mean()  

print(f"R2 Skoru: {r2:.4f}")
print(f"Max Hata: {max_error_celsius:.4f} °C")

R2 Skoru: 0.8982
Max Hata: 8.7397 °C


❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [6]:

r2 = cv_results["test_r2"].mean()

max_error_celsius = -cv_results["test_neg_max_error"].mean()

### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [7]:

model_mae = SGDRegressor(
    loss="epsilon_insensitive",
    epsilon=0,
    random_state=42
)

cv_results_mae = cross_validate(
    model_mae,
    X_scaled,
    y,
    cv=10,
    scoring=["r2", "neg_max_error"]
)




❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [8]:
r2_mae = cv_results_mae["test_r2"].mean()
max_error_mae = -cv_results_mae["test_neg_max_error"].mean()

print(f"MAE R2 Skoru: {r2_mae:.4f}")
print(f"MAE Max Hata: {max_error_mae:.4f} °C")

MAE R2 Skoru: 0.8763
MAE Max Hata: 10.8689 °C


## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

MSE ve MAE modellerinin R2 skorları birbirine yakın (0.90 vs 0.88) olsa da, MSE modeli maksimum hatada çok daha iyi performans gösteriyor

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [9]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-9.0.2, pluggy-1.6.0 -- /home/demet/.pyenv/versions/3.12.9/bin/python3.12
cachedir: .pytest_cache
rootdir: /home/demet/S16D4-S-loss-functions/tests
plugins: anyio-4.12.1, dash-4.0.0
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.15s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

